# 공통 분석 파이프라인 — 수치 탐색에서 해석적 증명까지

이 발표의 중심 질문은 다음과 같다. **수렴률 분석에 쓸 potential을 어떻게 찾고, 그 potential이 원하는 부등식을 만족한다는 것을 어떻게 증명할까?** GD에서 공통 분석 절차를 설명한 뒤, SGD에서 확률적 오차를 반영하고, averaging에서 최종 상한의 오차항을 줄이는 방법을 살펴본다.

발표는 다음 순서로 진행한다.

1. **공통 분석 파이프라인:** 계수 탐색 문제가 LMI와 SDP가 되는 이유
2. [**Example I · GD**](3_2_1_Figure_1_1A_GradientDescent.ipynb): Figure 1의 수치 탐색에서 Theorem 3까지
3. [**Example II · SGD**](3_2_2_Figure_2_2A_StochasticGradientDescent.ipynb): 같은 분석법에 분산을 반영하고 Figure 2와 Theorem 5로 연결
4. [**Averaging 확장**](Averaging_expansion_Theorem_6-7_Table_2.ipynb): 평균점과 oracle 평가점을 바꾸어 Theorem 6·7을 비교하고 Table 2로 종합

각 사례는 아래의 같은 다섯 단계를 따른다. 다음 단계는 앞 단계에서 얻은 조건이나 계수 수열을 이어받는다.

| 공통 단계 | 이 단계에서 답할 질문 | 다음 단계에서 사용할 결과 |
| --- | --- | --- |
| **1. 알고리즘과 가정** | 어떤 업데이트와 함수·oracle 조건을 분석하는가? | 한 단계의 업데이트 식과 사용할 부등식 |
| **2. Parameterized potential 설정** | 어떤 항을 묶고, 마지막에 무엇의 상한을 얻을 것인가? | 미지 계수, 한 단계 부등식, 초기·최종 조건 |
| **3. LMI/SDP로 계수 수치 탐색** | 가정을 만족하는 모든 함수와 oracle에 대해 부등식을 보장하는 계수를 어떻게 찾는가? | 함수값 항의 소거 조건과 LMI를 만족하는 수치 계수 수열 |
| **4. 계수 패턴 관찰과 단순화** | 어떤 계수를 0이나 상수로 두면 분석하기 쉬워지는가? | 명시적인 식으로 쓸 수 있는 potential 후보 |
| **5. 해석적 potential inequality와 수렴률** | 선택한 potential이 각 반복 단계에서 원하는 부등식을 만족하는 이유는 무엇인가? | 계수 점화식, 한 단계 부등식의 증명, 최종 상한과 수렴률 |

GD와 SGD 노트북에서는 각 알고리즘을 위의 다섯 단계에 따라 분석한다. Averaging 노트북에서는 두 평균법을 같은 절차로 각각 분석한 뒤, 함수값 상한과 수렴률을 비교한다.

## 1단계 · 알고리즘과 가정

공통으로 $f$는 $L$-smooth convex 함수이고, $x_\star$는 최적점, $f_\star=f(x_\star)$이다. 먼저 알고리즘의 업데이트를 고정한다. GD에서는

$$x_{k+1}=x_k-\frac1L\nabla f(x_k)$$

를 사용하고, SGD에서는 확률적 gradient를 반환하는 oracle을 사용한다.

$$x_{k+1}^{(i)}=x_k-\delta_kG(x_k;i),\qquad
\mathbb E_iG(x_k;i)=\nabla f(x_k),\qquad
\mathbb E_i\|G(x_k;i)-\nabla f(x_k)\|^2\le\sigma^2.$$

$\mathbb E_i$는 현재 상태까지의 정보를 고정한 조건부 기대값이다. 수치 실험에서는 oracle이 유한 개의 값 중 하나를 반환하도록 설정한다. 각 SDP를 풀 때 step-size와 업데이트 식은 고정되어 있다.

| 사례 | 앞 사례에서 이어받는 것 | 새로 달라지는 것 |
| --- | --- | --- |
| GD | 공통 분석법을 처음 구성 | 정확한 gradient를 사용하는 업데이트 |
| SGD | 함수 클래스와 potential·Gram·interpolation·LMI 구성 방식 | 조건부 기대값과 분산 상한 |
| SGD with averaging | SGD의 함수·oracle 가정과 업데이트 | SGD 갱신 후 반복점들의 평균 $z_k$에서 함수값을 평가 |
| Primal averaging | 같은 함수·oracle 가정과 평균점의 함수값을 분석하는 목표 | 평균점 $y_{k+1}$을 먼저 정하고 그 점에서 oracle을 평가 |

따라서 averaging으로 넘어갈 때도 동일한 분석법을 사용한다. 바뀐 업데이트가 한 단계 부등식과 오차 계수에 어떤 차이를 만드는지가 새 질문이다.

## 2단계 · Parameterized potential 설정

GD와 SGD에서는 거리 제곱, gradient norm 제곱, 교차항, 함수값으로 이루어진 potential을 사용한다.

$$\phi_k^f=
a_k\|x_k-x_\star\|^2+b_k\|\nabla f(x_k)\|^2
+2c_k\langle\nabla f(x_k),x_k-x_\star\rangle
+d_k(f(x_k)-f_\star).$$

평균법에서는 함수값과 gradient 관련 항을 $z_k$ 또는 $y_k$에서 평가하고, SGD 반복점 $x_k$의 거리항을 별도로 둔다. 알고리즘과 경계조건으로 고정한 계수 외에, 나머지 계수들을 한 단계의 potential 부등식을 만족하도록 찾는다.

$$\boxed{\mathbb E_i\phi_{k+1}^f\le\phi_k^f+e_k\sigma^2.}$$

GD에서는 이 부등식에서 기대값을 생략하고 오차항을 $0$으로 둔다. 초기·최종 potential은 마지막에 얻고 싶은 상한에 맞춰 정한다.

| 사례 | 최종 potential에 포함할 항 | 그 계수의 역할 |
| --- | --- | --- |
| GD | $b_N\|\nabla f(x_N)\|^2$ | gradient norm 제곱 상한의 분모 |
| SGD | $d_N(f(x_N)-f_\star)$ | 마지막 반복점의 함수값 상한의 분모 |
| Averaging·Primal averaging | $D_N(f(z_N)-f_\star)$ 또는 $D_N(f(y_N)-f_\star)$ | 평균점의 함수값 상한의 분모 |

평균법의 함수값 계수는 $D_k$로 쓴다. Theorem 6에서 평균을 만드는 가중치 $d_k$와 구분하기 위한 표기다. 각 노트북은 수식의 계수와 코드 변수의 대응도 함께 설명한다.

이 단계에서 **어떤 potential을 탐색할지와 어떤 상한을 개선할지**가 정해진다. 다음 단계에서는 이 조건을 계산 가능한 제약으로 바꾼다.

## 3단계 · LMI/SDP로 계수 수치 탐색

### 벡터에 대한 이차식을 Gram 행렬의 선형식으로 바꾸기

한 단계에서 필요한 실제 벡터를 $P$의 열로 모으고, 그 내적을

$$G=P^\top P\succeq0$$

로 표현한다. GD에서는 $P=[x_k-x_\star\quad\nabla f(x_k)\quad\nabla f(x_{k+1})]$이다. SGD와 평균법에서는 oracle 결과와 다음 점의 gradient 등 필요한 열을 추가한다.

$G$의 성분이 벡터 사이의 내적이므로, 거리 제곱과 gradient norm 제곱 같은 항은 $G$의 선형식으로 바뀐다. $G\succeq0$인 이유는 모든 벡터 $u$에 대해 $u^\top Gu=\|Pu\|^2\ge0$이기 때문이다. 함수값들은 별도의 열벡터 $\mathbf F$에 모은다.

세 종류의 행렬은 역할이 다르다.

| 행렬 | 의미 | 요구하는 조건 |
| --- | --- | --- |
| $G=P^\top P$ | 실제 벡터들의 내적 | $G\succeq0$ |
| $Q_k=\begin{pmatrix}a_k&c_k\\c_k&b_k\end{pmatrix}$ | potential의 이차항 계수 | 대칭성; 중간 상태에서 별도의 PSD 조건은 일반적으로 두지 않음 |
| $R_k$ | potential 차이에 가정을 나타내는 부등식들을 가중합해 더했을 때의 행렬 | $R_k\preceq0$ |

$G$는 Gram 행렬을, $G(x;i)$는 stochastic oracle을 뜻한다. 각 노트북의 굵은 벡터 기호는 실제 벡터를 $P$의 열벡터들의 선형결합으로 나타낼 때 쓰는 계수벡터를 뜻한다.

### Interpolation과 분산 조건을 더해 LMI 만들기

함수 클래스의 성질은 각 점 쌍의 interpolation 부등식으로 반영한다.

$$\mathcal I_{p,q}=
f(x_p)-f(x_q)-\langle\nabla f(x_q),x_p-x_q\rangle
-\frac1{2L}\|\nabla f(x_p)-\nabla f(x_q)\|^2\ge0.$$

확률적 방법의 분산 조건도 같은 Gram 행렬의 선형식
$\operatorname{Tr}(GA_{\rm var})\le\sigma^2$로 표현한다. 여기서 $A_{\rm var}$는 oracle 출력과 그 평균의 차이에 대한 norm 제곱의 기대값을 Gram 행렬로 나타낼 때 쓰는 계수행렬이다.

$\Delta\phi_k=\mathbb E_i\phi_{k+1}^f-\phi_k^f$라고 쓰자. 보이고 싶은 식 $\Delta\phi_k-e_k\sigma^2\le0$의 좌변에 비음수 항들을 더한다.

$$\begin{aligned}
&\Delta\phi_k-e_k\sigma^2
+\sum_{p\ne q}\lambda_{p,q}\mathcal I_{p,q}
+e_k\bigl(\sigma^2-\operatorname{Tr}(GA_{\rm var})\bigr)\\
&\qquad=\Delta\phi_k
+\sum_{p\ne q}\lambda_{p,q}\mathcal I_{p,q}
-e_k\operatorname{Tr}(GA_{\rm var}),
\qquad \lambda_{p,q}\ge0,\quad e_k\ge0.
\end{aligned}$$

오른쪽 식이 항상 0 이하임을 보이면 원래의 potential 부등식이 성립한다. 이 식은 Gram 행렬과 함수값에 대한 선형식이므로

$$\operatorname{Tr}(GR_k)+\text{함수값의 선형항}$$

으로 모을 수 있다. 이 식에서 함수값 항의 계수를 모두 $0$으로 두고 $R_k\preceq0$을 요구하면, 모든 $G\succeq0$와 함수값 벡터에서 그 식이 0 이하가 된다.

$$\boxed{\text{함수값 계수}=0,\qquad
R_k\preceq0,\qquad\lambda_{p,q}\ge0,\qquad e_k\ge0.}$$

이 가운데 $R_k\preceq0$이 **LMI**다. 업데이트를 고정했으므로 행렬의 성분은 미지의 potential 계수와 multiplier에 대해 affine하다. 각 반복 단계의 등식과 LMI를 제약으로 두고 선형 목적함수를 정하면 **SDP**가 된다. 여기서 최적화하는 것은 potential의 계수와 multiplier다.

GD에서는 위 식에서 분산 관련 항을 생략한다. SGD와 두 평균법은 같은 구성에 분산 행렬항이 들어간다. 각 사례의 차이는 업데이트와 선택한 potential이 이 행렬에 어떻게 반영되는가에 있다.

### 같은 제약 구성에서 목적함수 선택하기

목적함수는 2단계에서 정한 최종 상한에 맞춘다.

| 사례 | SDP의 목적함수 | 그 이유 |
| --- | --- | --- |
| GD | $b_N$ 최대화 | gradient norm 제곱 상한의 분모를 키움 |
| SGD | 먼저 $d_N$ 최대화, 그 수준을 거의 유지하며 $\sum e_k$ 최소화 | 초기 거리 항과 분산 항에 곱해지는 계수를 차례로 줄임 |
| SGD with averaging | 먼저 $D_N$ 최대화, 그 수준을 거의 유지하며 $\sum e_k$ 최소화 | 평균점의 함수값 상한에서 초기 거리 항과 분산 항의 계수를 차례로 줄임 |
| Primal averaging | 정해진 $D_k$ 아래에서 $\sum e_k$ 최소화 | $D_{k+1}=D_k+L\delta_k$가 평균 업데이트에 이미 들어감 |

이처럼 목적함수는 각 사례에서 구하려는 상한에 따라 정한다. Primal averaging에서는 함수값 계수가 알고리즘으로 정해지므로, 단계별 오차 계수의 합을 최소화하는 SDP를 한 번 푼다.

이 단계의 결과는 정해진 반복 횟수에 대해 계산한 **potential 계수 수열**이다. 이제 그 수열에서 분석하기 쉬운 구조를 찾는다.

## 4단계 · 계수 패턴 관찰과 단순화

수치해에서 계수의 변화 양상을 살펴보고, potential을 단순화할 조건을 정한다. 그 조건을 적용한 SDP를 다시 풀어, 최종 상한이 얼마나 달라지는지 비교한다.

| 사례 | 계수에서 관찰하고 비교할 것 | 해석적 증명에 사용할 potential 후보 |
| --- | --- | --- |
| GD | 최종 $b_N$의 증가, 함수값 계수의 선형 형태, 교차항·거리항의 계수를 제한했을 때의 변화 | 거리항 계수를 고정하고 교차항을 없앤 potential |
| SGD | $d_N$과 오차 계수의 누적합 사이의 관계, 교차항·gradient norm 제곱 항을 없애고 거리항을 고정했을 때의 변화 | $\phi_k^f=(L/2)\|x_k-x_\star\|^2+d_k(f(x_k)-f_\star)$ |
| 두 평균법 | 평균점의 이차항을 없앤 경우, 최종 거리항의 선택, 함수값 계수와 단계별 오차 계수의 관계 | 평균점의 함수값과 $x_k$의 거리항으로 이루어진 potential |

계수를 제한하는 실험에는 초기·최종 조건이 함께 바뀌는 경우도 있다. 각 노트북은 그 차이를 밝힌 상태에서 수치해를 비교한다. 확률적 방법에서는 함수값 계수뿐 아니라 **오차 계수의 누적합을 함수값 계수로 나눈 비율**까지 비교한다.

마지막 단계에서는 이 potential의 계수를 명시적으로 정하고, 원하는 부등식이 각 반복 단계에서 성립함을 증명한다.

## 5단계 · 해석적 potential inequality와 수렴률

단순화한 potential의 계수를 점화식이나 명시적 수열로 정하고, 사용할 interpolation 부등식과 비음수 multiplier를 선택한다. 가중합을 전개했을 때 함수값 항이 소거되거나 potential 차이로 모이고, 남은 제곱항의 부호가 정해지는 이유를 증명한다.

3단계에서는 조건을 만족하는 계수를 수치적으로 구했다. 여기서는 multiplier를 명시적인 식으로 제시하고, 함수값 항의 소거 조건과 $R_k\preceq0$이 각 반복 단계에서 성립함을 증명한다.

| 정리 | 5단계에서 증명할 내용 |
| --- | --- |
| Theorem 3 | 세 부등식의 가중합을 GD potential의 차이로 만드는 계수 점화식 |
| Theorem 5 | 조건부 기대값을 취한 SGD potential의 변화량을 분산항으로 제한하는 과정 |
| Theorem 6 | 평균 업데이트 식을 대입하고 완전제곱식으로 정리해 오차 계수를 구하는 과정 |
| Theorem 7 | 평균점에서 oracle을 평가할 때 교차항이 소거되어 더 단순한 오차 계수를 얻는 과정 |

한 단계 부등식을 증명한 뒤에는 기대값을 취하고 합하여

$$\mathbb E\phi_N^f
\le\phi_0^f+\sigma^2\sum_{k=0}^{N-1}e_k$$

를 얻는다. GD에서는 기대값과 오차항을 생략한다. 선택한 최종 potential에서 비음수 항을 버리면, 2단계에서 목표로 삼은 gradient norm 제곱 또는 함수값의 상한이 남는다.

이제 2단계에서 정한 목표에 맞는 상한을 얻었다. 이 상한이 반복 횟수에 따라 어떻게 변하는지를 살펴보면 된다.

## 발표 전체를 잇는 질문

이제 같은 다섯 단계를 따라 사례를 이어 간다.

- **GD → SGD:** GD에서 만든 Gram·interpolation·LMI 구성에 조건부 기대값과 분산 조건이 들어오면, 새로 어떤 계수가 필요하고 목적함수는 어떻게 달라지는가?
- **SGD → Averaging:** 함수 클래스와 oracle 가정을 유지하면서 평균점을 출력하면, 최종 함수값 상한의 오차항을 얼마나 줄일 수 있는가?
- **Averaging → Primal averaging:** 평균을 만드는 순서와 oracle 평가점을 바꾸면, 같은 분석법으로 구한 오차 계수는 어떻게 달라지는가?
- **두 평균법 → Table 2:** 앞에서 구한 명시적 계수들을 합했을 때, step-size 감소 속도는 초기 거리 항과 분산 항의 균형을 어떻게 바꾸는가?

Table 2 비교에서는 앞에서 구한 계수의 유한 합과 점근적 차수를 분석한다. 따라서 averaging 노트북의 마지막 부분에서는 두 평균법의 분석 결과를 앞선 SGD 결과와 함께 비교한다.

다음은 [**Example I · Gradient descent**](3_2_1_Figure_1_1A_GradientDescent.ipynb)다. 첫 사례에서 다섯 단계의 분석 과정을 살펴본 뒤, 이후 사례에서는 추가되거나 달라지는 부분에 주목한다.

---

참조 범위: 논문 §3.2.1·§3.2.2, Figure 1·2, Theorem 3·5·6·7 및 Table 2. 각 사례 노트북에 해당 수식·참조 코드·저장된 수치 결과와 상세 증명을 배치했다.